# BuyOrWait — Small-Sample EDA

Exploratory analysis on a subset of the Kaggle *100 Million+ Steam Reviews* dataset. This notebook documents the observations that motivated the two core metric designs:

1. **Playtime weighting** — a large share of reviews come from users with very little playtime; raw positive rate is dominated by low-information votes.
2. **Time decay (90-day half-life)** — review volume and sentiment shift sharply over a game's life (patches, controversies, sales), so old sentiment should not outvote recent sentiment.
3. **Volume-spike + z-score bombing detection** — review-bombing episodes show up as simultaneous spikes in daily volume and daily negative rate, which is why the alert requires both conditions.

> **Data is not committed to this repo** (see `.gitignore`). Run `pipeline/convert_to_parquet.py` first, then execute this notebook on the same machine. `DATA_DIR` defaults to `../slim_parquet`.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = os.environ.get("DATA_DIR", "../slim_parquet")
N_PARTS = 3  # small sample: first few part files (~a few million rows)

files = sorted(glob.glob(os.path.join(DATA_DIR, "part_*.parquet")))[:N_PARTS]
assert files, f"No part_*.parquet under {DATA_DIR} — run pipeline/convert_to_parquet.py first."
df = pd.read_parquet(files)
print(f"{len(df):,} rows from {len(files)} part file(s)")
df.head()

In [ ]:
# Schema, dtypes and null counts — the slim Parquet layer keeps only metric columns
pd.DataFrame({"dtype": df.dtypes.astype(str), "nulls": df.isna().sum(), "sample": df.iloc[0]})

## Why playtime weighting?

Playtime at review time is extremely right-skewed: many reviews are written after very little play. `log1p` compresses the scale so that a 1,000-hour reviewer does not drown out everyone else, while a 10-minute reviewer contributes close to zero weight.

In [ ]:
hours = df["playtime_at_review"] / 60.0
print(hours.describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1))
print(f"\nReviews with < 2h playtime: {(hours < 2).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
hours.clip(upper=100).hist(bins=60, ax=axes[0])
axes[0].set_title("Playtime at review (hours, clipped at 100)")
np.log1p(df["playtime_at_review"]).hist(bins=60, ax=axes[1])
axes[1].set_title("log1p(playtime_minutes) — the weight base")
plt.tight_layout()

## Why time decay?

Daily positive rate is not stationary — it moves with patches and controversies. A score that averages all history equally reports yesterday's game, not today's.

In [ ]:
df["date"] = pd.to_datetime(df["timestamp_created"], unit="s").dt.floor("D")

# Pick the highest-volume game in the sample and plot its daily sentiment
top_app = df["appid"].value_counts().idxmax()
g = (df[df["appid"] == top_app]
     .groupby("date")["voted_up"].agg(["size", "mean"])
     .rename(columns={"size": "n", "mean": "pos_rate"}))

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
g["pos_rate"].rolling(7, min_periods=1).mean().plot(ax=axes[0])
axes[0].set_title(f"appid {top_app}: 7d rolling positive rate — sentiment is far from constant")
g["n"].plot(ax=axes[1])
axes[1].set_title("Daily review volume — spikes mark releases, sales, controversies")
plt.tight_layout()

## Why dual-condition bombing detection?

On low-volume days a single negative review moves the daily negative rate enormously — a z-score alone would flood the alert list with tiny-sample noise. Requiring daily volume > 2× the 30-day rolling average as well keeps only episodes where *many* people showed up to complain at once.

In [ ]:
daily = (df.groupby(["appid", "date"])["voted_up"]
         .agg(["size", "mean"]).rename(columns={"size": "n", "mean": "pos_rate"})
         .reset_index())
daily["neg_rate"] = 1 - daily["pos_rate"]

small = daily[daily["n"] <= 3]
print(f"Game-days with <= 3 reviews: {len(small) / len(daily):.1%} of all game-days")
print(f"Their neg_rate std: {small['neg_rate'].std():.3f} vs "
      f"{daily.loc[daily['n'] > 30, 'neg_rate'].std():.3f} for game-days with > 30 reviews")

## Takeaways → metric design

- `wᵢ = log1p(playtime_at_review) × exp(−age_days / 90)` — playtime damps low-information votes, the 90-day half-life makes recent sentiment dominate.
- Bombing alert = neg-rate z-score > 3 (vs 30-day rolling baseline) **and** daily volume > 2× the 30-day rolling mean.
- The full 114M-row recalculation of these metrics is what `pipeline/pipeline.py` benchmarks on CPU vs GPU.